# YGOPRODeck Artwork Downloader (Google Drive Version)
Downloads both HD and SD images concurrently, zips them, and saves the zip directly to your Google Drive.

In [ ]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
import shutil
from google.colab import drive

# ==========================================
# CONFIGURATION
# ==========================================
TARGET_HD = True   # Download High Definition images (~421x614)
TARGET_SD = True   # Download Standard Definition images (~177x254)
LANGUAGE = ""      # E.g., "es", "fr". Leave empty for English.
SAVE_TO_DRIVE = True # Mounts Google Drive and copies the zip there
# ==========================================

if SAVE_TO_DRIVE:
    print("Requesting access to Google Drive...")
    drive.mount('/content/drive')

BASE_DIR = "/content/data/base"
PICS_DIR = f"{BASE_DIR}/pics"
API_URL = "https://db.ygoprodeck.com/api/v7/cardinfo.php"
if LANGUAGE:
    API_URL += f"?language={LANGUAGE}"

os.makedirs(f"{PICS_DIR}/hd", exist_ok=True)
os.makedirs(f"{PICS_DIR}/sd", exist_ok=True)

print("1/3 Fetching card database from YGOPRODeck...")
headers = {"User-Agent": "ProjectKaiba/1.0"}
response = requests.get(API_URL, headers=headers)
response.raise_for_status()
data = response.json().get("data", [])

print("2/3 Parsing card data and checking existing files (Resume check)...")
download_queue = []

for card in data:
    card_id = card.get("id")
    images = card.get("card_images", [])
    if not images:
        continue
    
    hd_url = images[0].get("image_url")
    sd_url = images[0].get("image_url_small")
    
    # Queue HD image if requested and missing
    if TARGET_HD and hd_url:
        dest = f"{PICS_DIR}/hd/{card_id}.jpg"
        if not os.path.exists(dest) or os.path.getsize(dest) == 0:
            download_queue.append((hd_url, dest))
            
    # Queue SD image if requested and missing
    if TARGET_SD and sd_url:
        dest = f"{PICS_DIR}/sd/{card_id}.jpg"
        if not os.path.exists(dest) or os.path.getsize(dest) == 0:
            download_queue.append((sd_url, dest))

if not download_queue:
    print("3/3 All requested images are already downloaded!")
else:
    print(f"3/3 Starting parallel download of {len(download_queue)} missing images...")

    def download_image(task):
        url, dest = task
        try:
            r = requests.get(url, headers=headers, stream=True)
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        except Exception as e:
            pass # Ignore minor errors on individual images so the queue doesn't stop

    # Download in parallel using 30 threads
    with ThreadPoolExecutor(max_workers=30) as executor:
        list(tqdm(executor.map(download_image, download_queue), total=len(download_queue), desc="Downloading"))

    print("Download complete! Images are safely stored in", PICS_DIR)

# Create a zip file and move it to Google Drive
if len(download_queue) > 0 or os.path.exists(PICS_DIR):
    print("\nZipping the images...")
    shutil.make_archive('/content/pics', 'zip', PICS_DIR)
    
    if SAVE_TO_DRIVE:
        print("Copying zip file to your Google Drive...")
        shutil.copy('/content/pics.zip', '/content/drive/MyDrive/ygo_pics.zip')
        print("Done! You can now find 'ygo_pics.zip' in the root of your Google Drive.")
    else:
        print("Done! You can now download /content/pics.zip from the Colab file browser on the left.")
